In [42]:
# imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import difflib

In [43]:
# Load data frame in
df = pd.read_csv('tracks_features.csv')

# Clean and grab important columns
# audio_features = [
#     'explicit', 'danceability', 'energy', 'loudness', 'mode', 
#     'speechiness', 'acousticness', 'instrumentalness', 'valence', 'tempo', 'liveness'
# ]
audio_features = [
    'danceability', 'energy', 'loudness', 
    'speechiness', 'acousticness', 'instrumentalness', 'valence', 'liveness'
]
features_df = df[audio_features].copy()
# features_df['explicit'] = features_df['explicit'].astype(float)
# features_df['mode'] = features_df['mode'].astype(float)

# Scale it so everything is normalized
scaler = MinMaxScaler()
normalized_matrix = scaler.fit_transform(features_df)

In [44]:
def compute_cosine_similarity(vec1, vec2):
    """
    Computes the cosine similarity between two numeric arrays
    """
    v1 = np.array(vec1, dtype=float)
    v2 = np.array(vec2, dtype=float)
    
    dot_product = np.dot(v1, v2)
    
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
        
    return dot_product / (norm_v1 * norm_v2)

In [45]:
def random_song_simularity():
    '''
    Computes the cosine simularity between two random songs
    '''
    random_indices = df.sample(n=2).index
    song1_idx = random_indices[0]
    song2_idx = random_indices[1]

    vec1 = normalized_matrix[song1_idx]
    vec2 = normalized_matrix[song2_idx]

    if 'name' in df.columns and 'artists' in df.columns:
        song1_info = f"{df.loc[song1_idx, 'name']} by {df.loc[song1_idx, 'artists']}"
        song2_info = f"{df.loc[song2_idx, 'name']} by {df.loc[song2_idx, 'artists']}"
    else:
        song1_info = f"Song {song1_idx}"
        song2_info = f"Song {song2_idx}"

    similarity_score = compute_cosine_similarity(vec1, vec2)

    print(f"Comparing:")
    print(f"1. {song1_info}")
    print(f"2. {song2_info}")
    print(f"-" * 30)
    print(f"Normalized Cosine Similarity Score: {similarity_score:.4f}")

In [46]:
def recommend_top_5_random():
    """
    Grabs a random song and recommends the top 5 most similar songs
    based on cosine similarity of their normalized audio features.
    """
    # 1. Grab a random song index
    random_idx = df.sample(n=1).index[0]
    
    # The target vector needs to be a 2D array for sklearn's cosine_similarity
    target_vec = normalized_matrix[random_idx].reshape(1, -1)
    
    # 2. Get info for the target song
    if 'name' in df.columns and 'artists' in df.columns:
        target_info = f"{df.loc[random_idx, 'name']} by {df.loc[random_idx, 'artists']}"
    else:
        target_info = f"Song {random_idx}"
        
    print(f"Target Song:\n{target_info}")
    print("-" * 40)
    
    # 3. Compute cosine similarities between the target song and ALL songs simultaneously
    # This returns an array of scores corresponding to every row in the dataframe
    similarities = cosine_similarity(target_vec, normalized_matrix)[0]
    
    # 4. Get the indices of the top 6 highest scores 
    # np.argsort returns indices sorted lowest to highest. We take the last 6 and reverse ([::-1])
    top_indices = np.argsort(similarities)[-6:][::-1]
    
    # 5. Filter out the target song itself (which will have a score of ~1.0) and keep top 5
    recommended_indices = [idx for idx in top_indices if idx != random_idx][:5]
    
    # 6. Print the recommendations
    print("Top 5 Recommendations:")
    for i, idx in enumerate(recommended_indices, 1):
        if 'name' in df.columns and 'artists' in df.columns:
            song_info = f"{df.loc[idx, 'name']} by {df.loc[idx, 'artists']}"
        else:
            song_info = f"Song {idx}"
            
        sim_score = similarities[idx]
        print(f"{i}. {song_info} (Similarity: {sim_score:.4f})")


In [53]:
def recommend_top_5_by_index(target_idx):
    """
    Given a specific index in the dataframe, recommends the top 5 
    most similar songs based on cosine similarity.
    """
    if target_idx not in df.index:
        print(f"Error: Index {target_idx} not found in the dataset.")
        return
    
    # Get the target vector for sklearn's cosine_similarity
    target_vec = normalized_matrix[target_idx].reshape(1, -1)
    # print(f"Target Song Index: {target_idx}")
    # print(target_vec)
    # print(f"Target Song: {df.loc[target_idx, 'name']} by {df.loc[target_idx, 'artists']}")
    # print(df.loc[target_idx])

    similarities = cosine_similarity(target_vec, normalized_matrix)[0]
    
    # Get the indices of the top 6 highest scores
    top_indices = np.argsort(similarities)[-6:][::-1]
    
    # Filter out the target song itself and keep the top 5
    recommended_indices = [idx for idx in top_indices if idx != target_idx][:5]
    
    print("-" * 40)
    print("Top 5 Recommendations:")
    for i, idx in enumerate(recommended_indices, 1):
        if 'name' in df.columns and 'artists' in df.columns:
            song_info = f"{df.loc[idx, 'name']} by {df.loc[idx, 'artists']}"
        else:
            song_info = f"Song {idx}"
            
        sim_score = similarities[idx]
        print(f"{i}. {song_info} (Similarity: {sim_score:.4f})")

In [48]:
def get_song_index_by_name(target_name):
    """
    Attempts to find a song by name. Falls back to finding the 
    closest matching string if an exact match isn't found.
    Returns the index of the match.
    """
    # 1. Fast Path: Try exact match (case-insensitive)
    exact_matches = df[df['name'].astype(str).str.lower() == target_name.lower()]
    if not exact_matches.empty:
        match_idx = exact_matches.index[0]
        print(f"Found exact match: '{df.loc[match_idx, 'name']}' by {df.loc[match_idx, 'artists']}")
        return match_idx
        
    # 2. Medium Path: Try substring match (e.g. searching "Bohemian" finds "Bohemian Rhapsody")
    substring_matches = df[df['name'].astype(str).str.contains(target_name, case=False, na=False)]
    if not substring_matches.empty:
        match_idx = substring_matches.index[0]
        print(f"Found substring match: '{df.loc[match_idx, 'name']}' by {df.loc[match_idx, 'artists']}")
        return match_idx
        
    # 3. Slow Path: Try fuzzy matching (closest string) using difflib
    print(f"'{target_name}' not found directly. Searching 1.2M rows for the closest match... (this may take a few seconds)")
    
    # Drop NAs and get unique names to speed up the difflib search slightly
    valid_names = df['name'].dropna().unique().astype(str)
    
    # cutoff=0.4 means it will accept matches that are at least 40% similar
    close_matches = difflib.get_close_matches(target_name, valid_names, n=1, cutoff=0.4)
    
    if close_matches:
        best_match_name = close_matches[0]
        # Get the index of this matched name
        match_idx = df[df['name'] == best_match_name].index[0]
        print(f"Found closest match: '{df.loc[match_idx, 'name']}' by {df.loc[match_idx, 'artists']}")
        return match_idx
    else:
        print(f"Could not find any close matches for '{target_name}'.")
        return None

In [56]:
def get_song_index_by_name_and_artist(target_name, target_artist):
    """
    Attempts to find a song by matching both the song name and the artist name.
    Returns the index of the match.
    """
    # 1. Fast Path: Exact name match AND artist substring match (case-insensitive)
    name_match = df['name'].astype(str).str.lower() == target_name.lower()
    artist_match = df['artists'].astype(str).str.contains(target_artist, case=False, na=False, regex=False)
    
    exact_matches = df[name_match & artist_match]
    
    if not exact_matches.empty:
        match_idx = exact_matches.index[0]
        print(f"Found exact match: '{df.loc[match_idx, 'name']}' by {df.loc[match_idx, 'artists']}")
        return match_idx
        
    # 2. Medium Path: Substring name match AND artist substring match
    name_sub_match = df['name'].astype(str).str.contains(target_name, case=False, na=False, regex=False)
    substring_matches = df[name_sub_match & artist_match]
    
    if not substring_matches.empty:
        match_idx = substring_matches.index[0]
        print(f"Found substring match: '{df.loc[match_idx, 'name']}' by {df.loc[match_idx, 'artists']}")
        return match_idx
        
    # If no matches are found
    print(f"Could not find any matches for '{target_name}' by '{target_artist}'.")
    return None

def recommend_from_song_and_artist(target_name, target_artist):
    """
    Wrapper function: Takes a user input song name and artist, finds the closest 
    song index, and prints recommendations.
    """
    print(f"Search Query: '{target_name}' by '{target_artist}'")
    
    song_idx = get_song_index_by_name_and_artist(target_name, target_artist)
    
    if song_idx is not None:
        # Re-using the recommendation function already present in your notebook
        recommend_top_5_by_index(song_idx)

In [49]:
def recommend_from_song_name(target_name):
    """
    Wrapper function: Takes a user input string, finds the closest 
    song index, and prints recommendations.
    """
    print(f"Search Query: '{target_name}'")
    
    song_idx = get_song_index_by_name(target_name)
    
    if song_idx is not None:
        recommend_top_5_by_index(song_idx)

In [50]:
random_song_simularity()

Comparing:
1. Prism by ['Environmental Sound Foundation']
2. For No One - 2007 Remaster by ['Emmylou Harris']
------------------------------
Normalized Cosine Similarity Score: 0.5308


In [51]:
recommend_top_5_random()

Target Song:
Lock and Key by ['Bessie Smith']
----------------------------------------
Top 5 Recommendations:
1. Christmas Time Is Here by ['Nichole Nordeman'] (Similarity: 0.9997)
2. Electronic Herd by ['Rachel Nelson'] (Similarity: 0.9996)
3. Kiss of Fire by ['Maria Volonte'] (Similarity: 0.9996)
4. Please Don't Let Me Go by ['Big Mike'] (Similarity: 0.9995)
5. I'll Buy You a Star by ['Vance Gilbert'] (Similarity: 0.9995)


In [ ]:
# Try an exact match
recommend_from_song_name("I'm Still Standing")


Search Query: 'I'm still standing'
Found exact match: 'I'm Still Standing' by ['Vitamin String Quartet']
----------------------------------------
Top 5 Recommendations:
1. Circles by ['JJ Grey & Mofro', 'Mofro'] (Similarity: 0.9993)
2. My Baby's Sweeter by ['Barbecue Bob & The Spareribs'] (Similarity: 0.9991)
3. Raga: Mishra Maloo (Gat in Vilambit Teental) by ['Shahid Parvez', 'Enayet Hossain'] (Similarity: 0.9989)
4. Offside Blues by ['Bob Degen'] (Similarity: 0.9989)
5. Two Hearts by ['Chris Isaak'] (Similarity: 0.9987)


In [57]:
recommend_from_song_and_artist("I'm Still Standing", "Elton John")

Search Query: 'I'm Still Standing' by 'Elton John'
Could not find any matches for 'I'm Still Standing' by 'Elton John'.
